In [1]:
import os
# os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

import sys

sys.path.insert(0, "../..")

In [2]:
import chex
import dill
import jax
import jax.numpy as jnp
import json
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import scipy
import torch
import treescope

from flax import nnx
from torch.utils.data import DataLoader
from typing import Any, NamedTuple
from typing_extensions import Protocol, runtime_checkable

from src.dataset import get_iter
from src.datasets.sum import Addition
from src.decoding import make_autoregressive
from src.rollout import rollout
from src.utils import parse_dict
from src.verifier import make_compute_returns

# Penzai
from penzai import pz

import IPython

pz.ts.register_as_default()

# Optional automatic array visualization extras:
pz.ts.register_autovisualize_magic()
pz.enable_interactive_context()
pz.ts.active_autovisualizer.set_interactive(pz.ts.ArrayAutovisualizer())


In [3]:
base_path = "/home/bryanpu1/projects/parallel_vs_serial/scaling_jax/results"
algo_name = "addition-max_int_64-no_eos-reset_immediately"
# run_name = "opc_ppo-scratch_1M-traj_improvement-0_cot_tokens-8x8-01-26-26_06_57_54-72a716da-5119-4bad-92d3-d498cbe06d27"
# run_name = "opc_ppo-scratch_1M-0_cot_tokens-8x8-01-26-26_07_00_51-eaee1709-185a-4366-938f-752d2aefd1ea"
# run_name = "ppo_kl-scratch_1M-0_cot_tokens-8x8-01-23-26_11_05_40-93cc131e-8f27-4175-88b1-51de541e4d0b"
# run_name = "opc_ppo-scratch_1M-include_all_in_len_examples-save_when_multiple_attempts-traj_improvement-0_cot_tokens-8x8-02-02-26_07_10_26-0d777242-30c2-4a8a-8ad9-31fcb7eff01b"
run_name = "opc_ppo-scratch_1M-discourage_first_reset-include_all_in_len_examples-save_when_multiple_attempts-traj_improvement-0_cot_tokens-8x8-02-03-26_10_19_14-c2e9addf-156e-4b77-b352-d03eaade5e43"
run_name = "opc_ppo-scratch_1M-zero_on_reset_discourage_first_reset-include_all_in_len_examples-save_when_multiple_attempts-traj_improvement-0_cot_tokens-8x8-02-06-26_03_30_40-ca2237d4-d1e0-425d-bd5a-4a4a50023adb"

learner_path = os.path.join(base_path, algo_name, run_name)

In [4]:
eval_seed = 42
num_evals = 1
max_decode_len = 200
num_bits = 12
max_int = 2 ** num_bits
max_batch_size = 1
checkpoint_i = -1

In [5]:
config_dict = json.load(open(os.path.join(learner_path, "config.json"), "r"))
half_precision = config_dict["half_precision"]
dtype = jnp.bfloat16 if half_precision else jnp.float32

# Get dataset
train_max_int = config_dict["dataset_kwargs"]["max_int"]
config_dict["dataset_kwargs"]["max_int"] = 4096
config_dict["gamma"] = 0.99
config = parse_dict(config_dict)
dataset_kwargs = config.dataset_kwargs

num_checkpoints = len(os.listdir(os.path.join(learner_path, "models")))

if config.dataset_name == "curriculum":
    train_num_bits = int(np.ceil(np.log2(dataset_kwargs.datasets[-1]["dataset_kwargs"]["max_int"])))
    train_val_ratio = dataset_kwargs.datasets[-1]["dataset_kwargs"]["train_val_ratio"]
    predict_eos = dataset_kwargs.datasets[-1]["dataset_kwargs"]["predict_eos"]
    num_cot_tokens = dataset_kwargs.datasets[-1]["dataset_kwargs"]["num_cot_tokens"]
    right_to_left = dataset_kwargs.datasets[-1]["dataset_kwargs"]["right_to_left"]
    correctness_aware = dataset_kwargs.datasets[-1]["dataset_kwargs"]["correctness_aware"]
    carry_registers = dataset_kwargs.datasets[-1]["dataset_kwargs"]["carry_registers"]
    train_max_decode_len = dataset_kwargs.datasets[-1]["dataset_kwargs"]["context_len"]
else:
    train_num_bits = int(np.ceil(np.log2(dataset_kwargs.max_int)))
    train_val_ratio = dataset_kwargs.train_val_ratio
    predict_eos = dataset_kwargs.predict_eos
    num_cot_tokens = dataset_kwargs.num_cot_tokens
    right_to_left = dataset_kwargs.right_to_left
    correctness_aware = dataset_kwargs.correctness_aware
    carry_registers = getattr(dataset_kwargs, "carry_registers", False)
    train_max_decode_len = dataset_kwargs.context_len

dataset = Addition(
    context_len=max_decode_len,
    max_int=max_int,
    train=True,
    seed=eval_seed,
    sequence_type="question_only",
    train_val_ratio=1.0,
    right_to_left=right_to_left,
    num_repeats=1,
    shuffle=True,
    exact=True,
    predict_eos=predict_eos,
    num_cot_tokens=num_cot_tokens,
    p_curriculum=0.0,
    p_inject_noop=0.0,
    max_noops=0,
    noop_as_pad=False,
    reverse_curriculum=False,
    correctness_aware=correctness_aware,
    carry_registers=carry_registers,
)

EOS TOKEN: 6
TOKEN MAP: {0: 0, 1: 1, 2: 2, 3: 3, 6: 6, 4: 4, 5: 5}


In [6]:
num_pairs = max_int * max_int
batch_size = min(num_pairs, max_batch_size)

out_dim = int(dataset.output_space.n)
data_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    num_workers=0,
)
data_iter = get_iter(data_loader, None, dtype)

In [7]:
last_step = sorted(os.listdir(os.path.join(learner_path, "models")))[checkpoint_i]
train_state = dill.load(
    open(os.path.join(learner_path, "models", last_step), "rb")
)

model = nnx.merge(
    train_state.graphdef,
    train_state.params,
    train_state.rest,
)
model.set_attributes(deterministic=False, decode=False)

In [8]:
batch = next(data_iter)
batch = {
    k: np.repeat(v, num_evals, axis=0)
    for k, v in batch.items()
}

In [9]:
batch

{'sequence': array([[0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 2, 1, 1, 1, 1, 0, 0, 1, 1, 0,
         1, 1, 1, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6]]),
 'target': array([[3, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6]]),
 'mask': array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1.]], dtype=float32),
 'pointer_correct': array([1]),
 'question_len': array([25]),
 'solution_len': array([14])}

In [10]:
model = nnx.merge(
    train_state.graphdef,
    train_state.params,
    train_state.rest,
)

rng = jax.random.PRNGKey(eval_seed)
rng, rollout_rng = jax.random.split(rng)

# Sample next batch and evaluate
_, init_cache = make_autoregressive(
    model,
    max_decode_len=max_decode_len,
    batch_size=batch_size,
    embed_dim=config_dict["model_config"]["model_kwargs"]["embed_dim"],
    dtype=dtype,
    eval_mode=True,
)
cache = init_cache()
graphdef, _, rest = nnx.split(model, nnx.Cache, ...)

rollout_res = rollout(
    graphdef,
    cache,
    rest,
    rollout_rng,
    batch,
    eos_token=dataset.eos_token_id,
    deterministic=int(num_evals == 1),
    correct_aware_shift=dataset.correctness_aware_tokens_offset,
    max_token_id_to_shift=dataset.max_token_id_to_shift,
)

In [11]:
model = nnx.merge(
    train_state.graphdef,
    train_state.params,
    train_state.rest,
)
model.set_attributes(deterministic=False, decode=False)

res = model({
    "sequence": rollout_res.observations
})

In [12]:
intermediates = nnx.pop(model, nnx.Intermediate)

In [13]:
rollout_res.success

Array([False], dtype=bool)

In [14]:
(batch["question_len"] + rollout_res.response_length).astype(int).shape

(1,)

In [15]:
res.shape, rollout_res.observations.shape

((1, 200, 4), (1, 200))

In [16]:
reset_idxes = np.where(rollout_res.observations[0] == dataset.reset_token_id)[0]
if rollout_res.success[0].item():
    reset_idxes = np.concatenate((reset_idxes, [np.argmax(rollout_res.observations[0] == dataset.eos_token_id)]))
reset_idxes[1:] - reset_idxes[:-1]

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 2, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2])

In [17]:
treescope.render_array(
    np.vstack((
        rollout_res.observations,
        np.argmax(res, axis=-1),
    ))
)


<Arrayviz rendering>

In [18]:
treescope.render_array(
    res[0].T[:, :batch["question_len"].item() + rollout_res.response_length.item()]
)


<Arrayviz rendering>

In [19]:
attn_weights = []
input_length = np.where(batch["sequence"][0] == dataset.eos_token_id)[0][0]
for layer_i in intermediates["gpt"]["gpt"]["layers"]:
    attn_weights.append(np.mean(
        intermediates["gpt"]["gpt"]["layers"][layer_i]["attention"]["attention_weights"].value[0][0],
        axis=0,
    )[:batch["question_len"].item() + rollout_res.response_length.item(), :batch["question_len"].item() + rollout_res.response_length.item()])

attn_weights = pz.nx.wrap(
    attn_weights
).tag("layer", "q", "k")
pz.ts.render_array(
    attn_weights,
    rows=["q"],
    columns=["k"]
)

<Arrayviz rendering>

In [20]:
attn_weights = []
for layer_i in intermediates["gpt"]["gpt"]["layers"]:
    attn_weights.append(
        intermediates["gpt"]["gpt"]["layers"][layer_i]["attention"]["attention_weights"].value[0][0, :, :batch["question_len"].item() + rollout_res.response_length.item(), :batch["question_len"].item() + rollout_res.response_length.item()]
    )

attn_weights = pz.nx.wrap(
    attn_weights
).tag("layer", "head", "q", "k")
pz.ts.render_array(
    attn_weights,
    rows=["q"],
    columns=["k"]
)

<Arrayviz rendering>

In [21]:
config_dict["gamma"] = 0.99
compute_returns = make_compute_returns(
    parse_dict(config_dict),
    dataset.eos_token_id,
    dataset.reset_token_id,
    dataset.token_map,
)

In [22]:
compute_returns(batch, rollout_res)

Array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.07071429,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.03534564,  0.03499219,  0.13996875,  0.13996875,  0.13996875,
         0.13996875,  0.13996875,  0.13856906, -0.17147921, -0.16976443,
        -0.16806678, -0.16638611, -0.16472226, -0.16307503, -0.16144428,
        -0.15982985, -0.15823156, -0.15664923, -0.15508275, -0.15353191,
        -0.15199661, -0.15047663, -0.14897187, -0.14748216, -0.14600733,
        -0.14454725, -0.1431018 , -0.14167078, -0.14025408, -0.13885152,
        -0.13746302, -0.13608839, -0.13472751, -0.13338023, -0.13204643,
        -0.13072596, -0.1294187 , -0.12812452, -0.12684327, -0.12557484,
        -0.12431911, -0.12307591, -0.12184515, -0.1206267 , -0.11942043,
        -0.11822623, -0.11704397, -0.11587353, -0.11471479, -0.11356765,
        -0.11243198, -0.11130765, -0.11019458, -0.10909264, -0.10800172,
        -0.1069217 , -0.10585248, -0.10479395, -0.10374602, -0.10270856,
        -0.10168147, -0.10066466, -0.09965801, -0.09866143, -0.09767482,
        -0.09669808, -0.09573109, -0.09477378, -0.09382605, -0.09288778,
        -0.09195891, -0.09103931, -0.09012893, -0.08922764, -0.08833537,
        -0.08745201, -0.0865775 , -0.08571172, -0.0848546 , -0.08400606,
        -0.083166  , -0.08233434, -0.081511  , -0.08069589, -0.07988892,
        -0.07909004, -0.07829914, -0.07751615, -0.07674099, -0.07597359,
        -0.07521385, -0.07446171, -0.07371709, -0.07297992, -0.07225012,
        -0.07152762, -0.07081234, -0.07010423, -0.06940318, -0.06870915,
        -0.05496732, -0.05441765, -0.05441765, -0.05387347, -0.06802206]],      dtype=float32)

In [23]:
config

namespace(logging_config=namespace(save_path='./results/addition-max_int_64-no_eos-reset_immediately',
                                   experiment_name='opc_ppo-scratch_1M-zero_on_reset_discourage_first_reset-include_all_in_len_examples-save_when_multiple_attempts-traj_improvement-0_cot_tokens-8x8',
                                   log_interval=1,
                                   checkpoint_interval=50000,
                                   validation_interval=500),
          model_config=namespace(architecture='InContextGPT',
                                 model_kwargs=namespace(num_blocks=4,
                                                        num_heads=8,
                                                        embed_dim=64,
                                                        widening_factor=4,
                                                        embedder_strategy='next_token',
                                                        shared_decoding=True,
                                                        use_sink_token=False)),
          optimizer_config=namespace(optimizer='adamw',
                                     lr=namespace(scheduler='constant_schedule',
                                                  scheduler_kwargs=namespace(value=0.0001)),
                                     opt_kwargs=namespace(weight_decay=0.0),
                                     max_grad_norm=1.0,
                                     mask_names=[]),
          half_precision=False,
          mesh=namespace(data=1, fsdp=1, tensor=1),
          learner='OffPolicyContextPPO',
          dataset_name='addition',
          dataset_kwargs=namespace(context_len=100,
                                   max_int=4096,
                                   train=True,
                                   train_val_ratio=1.0,
                                   sequence_type='question_only',
                                   num_repeats=1000000,
                                   right_to_left=True,
                                   shuffle=True,
                                   predict_eos=False,
                                   num_cot_tokens=0,
                                   p_curriculum=0,
                                   reverse_curriculum=False,
                                   correctness_aware=True,
                                   carry_registers=False),
          repeat=False,
          shuffle_buffer_size=100,
          num_workers=1,
          train_loss_config=namespace(objective='ppo',
                                      mdp_type='traj_improvement:length_bias_fix',
                                      clip_param=0.2,
                                      entropy=0.0),
          num_ppo_steps=3,
          gamma=0.99,
          buffer_size=1000000,
          max_seq_len=100,
          reward_type='negative_on_failure',
          num_rollouts_per_sample=8,
          batch_size=4,
          batch_size_buffer=4,
          num_epochs=1000000,
          num_updates_per_epoch=1,
          seeds=namespace(seed=0, learner_seed=42, data_seed=42),
          validation=[{'validation_name': 'greedy_success',
                       'dataset_name': 'addition',
                       'dataset_kwargs': {'context_len': 100,
                        'max_int': 64,
                        'train': True,
                        'train_val_ratio': 1.0,
                        'sequence_type': 'question_only',
                        'num_repeats': None,
                        'right_to_left': True,
                        'shuffle': False,
                        'exact': False,
                        'predict_eos': False,
                        'num_cot_tokens': 0,
                        'correctness_aware': True,
                        'carry_registers': False},
                       'shuffle_buffer_size': 100,
                       'num_workers': 1,
                       'batch_size': 100,
                       'see

In [24]:
batch

{'sequence': array([[0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 2, 1, 1, 1, 1, 0, 0, 1, 1, 0,
         1, 1, 1, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6]]),
 'target': array([[3, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6]]),
 'mask': array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1.]], dtype=float32),
 'pointer_correct': array([1]),
 'question_len': array([25]),
 'solution_len': array([14])}